# YOLOv8n Baseline Training, Dataset V2
## Overview
This notebook prepares Roboflow Dataset V2 and records the YOLOv8n baseline training setup for pork rasher packaging defect detection.
## Dataset Preparation
Images from the same source are identified using the filename prefix before `.rf.`. Overlapping sources are removed across train, validation and test splits. The cleaned data is checked before training.<br>
Dataset V2 contains four defect classes: loose-meat, packaging-error, twisted-meat and unsealed. No wrinkle holdout is applied.
## Files
*   Input dataset: dataset/pork_rasher_v2.zip
*   Prepared dataset: /content/pork_rasher_dataset_v2_clean (temporary Colab folder)
*   Preparation record: dataset/manifests/v2_preparation.json
*   Original baseline run: model_runs/yolov8n_v2_baseline
## Limitations
The source-overlap check uses JPG filenames and may miss duplicated images with different source identifiers. The preparation record does not confirm the training history of existing model weights.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import sys

# Dataset and saved model folders.
PROJECT_ROOT = Path("/content/drive/MyDrive/MMA3001_v2_folder")
ZIP_PATH = PROJECT_ROOT / "dataset" / "pork_rasher_v2.zip"
CLEAN_DIR = Path("/content/pork_rasher_dataset_v2_clean")
MANIFEST_PATH = PROJECT_ROOT / "dataset" / "manifests" / "v2_preparation.json"
RUNS_DIR = PROJECT_ROOT / "model_runs"
ORIGINAL_RUNS_DIR = Path("/content/drive/MyDrive/MMA3001_Project/model_runs")

# Find the shared cleaning code.
module_candidates = [
    Path.cwd() / "src",
    Path("/content/src"),
    Path("/content"),
    PROJECT_ROOT / "src",
    Path("/content/drive/MyDrive/MMA3001_v4_Project/src"),
]
module_candidates += sorted(Path("/content").glob("*/src"))
module_dir = next((p for p in module_candidates
                   if (p / "dataset_preparation.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError(
        "dataset_preparation.py not found. Copy it to the V2 project's src folder "
        "or use a cloned GitHub repository in /content/."
    )
sys.path.insert(0, str(module_dir))
from dataset_preparation import prepare_dataset, print_audit, assert_preparations_match

In [4]:
# Clean source overlaps using the shared module.
manifest = prepare_dataset(
    zip_path=ZIP_PATH,
    output_dir=CLEAN_DIR,
    expected_dataset_version=2,
    require_labels=True,
)
print("Cross-split overlaps removed:", manifest["removed_overlap_images"])
print_audit(manifest["audit"])

Cross-split overlaps removed: {'train': 9, 'valid': 1}

TRAIN: 1935 images; 362 backgrounds; 0 missing label files
  loose-meat: 65 boxes
  packaging-error: 521 boxes
  twisted-meat: 67 boxes
  unsealed: 1045 boxes

VALID: 119 images; 14 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 34 boxes
  twisted-meat: 3 boxes
  unsealed: 80 boxes

TEST: 80 images; 12 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 23 boxes
  twisted-meat: 4 boxes
  unsealed: 49 boxes


In [5]:
# Check the original V2 split counts and class order.
EXPECTED_IMAGES = {"train": 1935, "valid": 119, "test": 80}
EXPECTED_CLASSES = ["loose-meat", "packaging-error", "twisted-meat", "unsealed"]
actual_images = {s: d["images"] for s, d in manifest["audit"]["splits"].items()}
assert actual_images == EXPECTED_IMAGES, (actual_images, EXPECTED_IMAGES)
assert manifest["audit"]["class_names"] == EXPECTED_CLASSES
assert manifest["moved_to_validation"] == 0
print("V2 split counts and class order verified.")
print("Split fingerprint:", manifest["split_fingerprint_sha256"])

V2 split counts and class order verified.
Split fingerprint: 37574f1f67a94a0aa532cce668ed5c0e8c53b0629191a967498d2d3b8d42e711


In [6]:
import json

# Save or check the preparation record.
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
if MANIFEST_PATH.exists():
    assert_preparations_match(manifest, MANIFEST_PATH)
    print("Preparation record matches:", MANIFEST_PATH)
else:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    print("Preparation record saved:", MANIFEST_PATH)

Preparation record saved: /content/drive/MyDrive/MMA3001_v2_folder/dataset/manifests/v2_preparation.json


## Baseline Model Training
YOLOv8n was trained for 30 epochs at 640-pixel resolution, batch size 16 and seed 42. The previous run completed training on 1,935 images, with 119 images used for validation. New training is disabled by default; existing weights are retained.

In [8]:
# Keep the original trained weights unless a new run is required.
RETRAIN_BASELINE = False

if RETRAIN_BASELINE:
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise ImportError("Install Ultralytics with %pip install ultralytics") from exc

    model = YOLO("yolov8n.pt")
    results = model.train(
        data=str(CLEAN_DIR / "data.yaml"),
        epochs=30,
        imgsz=640,
        batch=16,
        device=0,
        seed=42,
        project=str(RUNS_DIR),
        name="yolov8n_v2_baseline_reproduced",
    )
    print("Training outputs:", results.save_dir)
else:
    print("Training skipped. Existing baseline weights are unchanged.")

Training skipped. Existing baseline weights are unchanged.
